In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
CHART_DIR = "charts"
sns.set_style("whitegrid")

In [3]:
df = pd.read_csv("delhi_business_scored.csv")
no_site = df[df["has_website"] == 0].copy()
insights = []

### INSIGHT1: Overall gap size

In [4]:
pct_no_website = (df["has_website"] == 0).mean() * 100
insights.append(f"1. {pct_no_website:.1f}% of all {len(df)} businesses tracked across Delhi have "f"no website -- a large addressable market for web development services.")

### INSIGHT2: Which category has the biggest gap?

In [5]:
gap_by_cat = df.groupby("category_searched")["has_website"].apply(lambda x: (x == 0).mean() * 100).sort_values(ascending=False)
insights.append(f"2. '{gap_by_cat.index[0]}' has the largest website gap "f"({gap_by_cat.iloc[0]:.0f}% have no site), while '{gap_by_cat.index[-1]}' has the "f"smallest ({gap_by_cat.iloc[-1]:.0f}%). Prioritize outreach to the high-gap categories.")

### INSIGHT 3: High-propensity, no-website businesses = best leads

In [6]:
strong_leads = no_site[no_site["website_propensity_score"] >= 0.7]
insights.append(f"3. {len(strong_leads)} businesses have no website but scored >=0.70 on the "f"website-propensity model -- meaning they look just like businesses that already "f"have one (similar rating, review volume, category, area). These are the strongest leads.")

###  INSIGHT 4: Best area to focus a sales campaign

In [7]:
area_leads = strong_leads.groupby("area").size().sort_values(ascending=False)
insights.append(f"4. '{area_leads.index[0]}' has the most high-propensity leads ({area_leads.iloc[0]}) "f"-- the most efficient area to start a door-to-door or local-ad outreach campaign.")

#### => LEAD SCORE = propensity weighted by business size (review_count)
#### => A business with high propensity AND high review_count is bigger / busier /
#### => likely to have more budget -- a better client than a small high-propensity one.

In [8]:
no_site["lead_score"] = (no_site["website_propensity_score"] * 0.7 + (no_site["review_count"].rank(pct=True)) * 0.3)
no_site = no_site.sort_values("lead_score", ascending=False)
hot_leads_cols = ["name", "area", "category_searched", "rating", "review_count","website_propensity_score", "lead_score"]
hot_leads = no_site[hot_leads_cols].head(200)
hot_leads.to_csv("hot_leads.csv", index=False)
print(f"\n Saved hot_leads.csv -- top 200 outreach-ready leads, ranked by lead_score")

insights.append(
    f"5. INNOVATION - Lead Score: combines website-propensity with business size "
    f"(review_count percentile) so outreach prioritizes established, busy businesses "
    f"over small ones -- better clients, same amount of sales effort. Top lead: "
    f"'{hot_leads.iloc[0]['name']}' in '{hot_leads.iloc[0]['area']}' "
    f"(lead score {hot_leads.iloc[0]['lead_score']:.2f})."
)

with open("insights.txt", "w") as f:
    f.write("DELHI WEB-DEV LEAD GENERATION - KEY INSIGHTS\n")
    f.write("=" * 55 + "\n\n")
    for line in insights:
        f.write(line + "\n\n")
print("\n" + "=" * 60)
print("INSIGHTS (also saved to insights.txt)")
print("=" * 60)
for line in insights:
    print("\n" + line)
 


 Saved hot_leads.csv -- top 200 outreach-ready leads, ranked by lead_score

INSIGHTS (also saved to insights.txt)

1. 49.0% of all 11784 businesses tracked across Delhi have no website -- a large addressable market for web development services.

2. 'hardware stores' has the largest website gap (74% have no site), while 'clinics' has the smallest (34%). Prioritize outreach to the high-gap categories.

3. 106 businesses have no website but scored >=0.70 on the website-propensity model -- meaning they look just like businesses that already have one (similar rating, review volume, category, area). These are the strongest leads.

4. 'Rohini, Delhi' has the most high-propensity leads (9) -- the most efficient area to start a door-to-door or local-ad outreach campaign.

5. INNOVATION - Lead Score: combines website-propensity with business size (review_count percentile) so outreach prioritizes established, busy businesses over small ones -- better clients, same amount of sales effort. Top lea

### % Without Website by Category

In [9]:
plt.figure(figsize=(11, 6))
sns.barplot(x=gap_by_cat.values, y=gap_by_cat.index, palette="rocket")
plt.title("% of Businesses With No Website, by Category", fontsize=13, fontweight="bold")
plt.xlabel("% with no website")
plt.ylabel("")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/09_no_website_by_category.png")
plt.close()

C:\Users\georg\AppData\Local\Temp\ipykernel_27724\2539548714.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=gap_by_cat.values, y=gap_by_cat.index, palette="rocket")


### Top 15 leads by lead_score

In [10]:
top15 = hot_leads.head(15).copy()
top15["label"] = top15["name"].str.slice(0, 25) + " (" + top15["area"].str.replace(", Delhi", "") + ")"
plt.figure(figsize=(11, 7))
sns.barplot(data=top15, x="lead_score", y="label", palette="flare")
plt.title("Top 15 Web-Dev Leads (Established Businesses, No Website)",
          fontsize=13, fontweight="bold")
plt.xlabel("Lead Score")
plt.ylabel("")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/10_top_leads.png")
plt.close()

C:\Users\georg\AppData\Local\Temp\ipykernel_27724\3548755090.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top15, x="lead_score", y="label", palette="flare")
